# Лабораторная работа 1: Регрессия

**Цель работы:** Пройти полный цикл решения задачи регрессии: от первичного анализа данных и простого baseline до сравнения нескольких моделей и анализа результатов. Достичь качества $RMSE < 0.70$ на тестовой выборке.

Дополнительная метрика - MAE.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

## 1. Знакомство с данными и первичный анализ

Загрузим локальные файлы `winequality-red.csv` и `winequality-white.csv` из текущей директории и объединим их в единый датасет. Добавим бинарный признак `is_red` (1 — красное вино, 0 — белое вино), так как органолептические и физико-химические свойства разных сортов заметно различаются.

In [ ]:
df_red = pd.read_csv('winequality-red.csv', sep=';')
df_white = pd.read_csv('winequality-white.csv', sep=';')

df_red['is_red'] = 1
df_white['is_red'] = 0

df = pd.concat([df_red, df_white], ignore_index=True)

print(
    f'Размер красного вина: {df_red.shape[0]} строк, белого вина:'
    f' {df_white.shape[0]} строк'
)
print(
    f'Размер объединенного датасета: {df.shape[0]} строк, {df.shape[1]}'
    ' столбцов'
)
print(f'Количество пропусков в данных: {df.isna().sum().sum()}')
display(df.head())
display(df.describe().T[['count', 'mean', '50%', 'min', 'max', 'std']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(data=df, x='quality', hue='is_red', ax=axes[0], palette='Set2')
axes[0].set_title('Распределение оценок качества вина (quality)')
axes[0].set_xlabel('Оценка качества (балл)')
axes[0].set_ylabel('Количество объектов')
axes[0].legend(title='Тип вина', labels=['Белое (0)', 'Красное (1)'])

corr_matrix = df.corr()
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    ax=axes[1],
    cbar=False,
    annot_kws={'size': 8},
)
axes[1].set_title('Матрица корреляций признаков')

plt.tight_layout()
plt.show()

### Выводы по первичному анализу данных:
1. **Масштаб данных и выбросы:** В датасете 6497 строк (1599 красного и 4898 белого вина), пропусков нет. Признаки измеряются в совершенно разных масштабах: например, плотность (`density`) держится около $0.99\text{–}1.04$, а общий диоксид серы (`total sulfur dioxide`) доходит до $440$. Кроме того, в остаточном сахаре (`residual sugar`) и диоксиде серы есть сильные выбросы (максимумы в десятки раз выше медианы). Значит, перед обучением линейной регрессии данные нужно будет нормализовать и отмасштабировать.
2. **Дисбаланс целевой переменной:** Оценки `quality` идут от 3 до 9 баллов, но распределены неравномерно — больше 75% всех вин получили средние оценки (5 или 6 баллов). Очень плохих (3–4) и отличных (8–9) вин в выборке мало, поэтому моделям будет трудно точно предсказывать такие крайние оценки.
3. **Главные зависимости (корреляции):**
   * Сильнее всего на оценку влияет **`alcohol`** ($r = 0.44$): чем выше градус, тем в среднем выше качество вина (крепкие вина обычно получаются из более спелого и качественного винограда). Этот признак точно будет самым важным для моделей.
   * Заметную отрицательную связь показывают **`density`** ($r = -0.31$) и **`volatile acidity`** (летучая кислотность, $r = -0.27$). Это логично: избыток летучих кислот дает неприятный запах уксуса и портит вкус вина.
   * Между самими признаками тоже есть сильные связи (мультиколлинеарность): например, чем больше спирта (`alcohol`), тем ниже плотность (`density`, $r = -0.69$), а свободный и общий диоксид серы тесно связаны друг с другом ($r = 0.72$). Чтобы линейную модель не «шатало» от зависимых признаков, для нее пригодится регуляризация (`Ridge`).

## 2. Подготовка выборок

Разделим данные на обучающую (`train`) и тестовую (`test`) части в соотношении 80% / 20%, зафиксировав `random_state = 42` и применив стратификацию по целевой переменной `quality`.

In [ ]:
X = df.drop(columns=['quality'])
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Размер обучающей выборки (Train): {X_train.shape[0]} объектов')
print(f'Размер тестовой выборки (Test):  {X_test.shape[0]} объектов')

### Обоснование разбиения данных:
* Данные разделены в пропорции **80% на обучение (5197 строк)** и **20% на тест (1300 строк)** с фиксацией `random_state = 42` для воспроизводимости. Такого объема `train` достаточно для обучения нелинейных моделей, а 1300 объектов в `test` дают устойчивую оценку итоговой ошибки.
* Использовано стратифицированное разбиение (`stratify=y`), чтобы сохранить одинаковые пропорции редких оценок качества (особенно баллов 3 и 9) в обеих выборках.
* Тестовая выборка используется только один раз для финального сравнения моделей. Если многократно подбирать параметры или признаки, ориентируясь на метрику на `test`, возникнет неявная утечка данных (переобучение под тестовую выборку), и оценка качества станет завышенной.

## 3. Метрики и baseline

* **MAE (Mean Absolute Error)** показывает, на сколько баллов `quality` в среднем по модулю ошибается модель. Все отклонения учитываются линейно с одинаковым весом, поэтому метрика легко интерпретируется и слабо реагирует на единичные выбросы.
* **RMSE (Root Mean Squared Error)** — корень из среднеквадратичной ошибки. Так как перед усреднением ошибки возводятся в квадрат, крупные промахи штрафуются непропорционально сильно (например, ошибка в 1 балл дает вклад 1, а ошибка в 3 балла — уже 9).
* Из-за квадратичного штрафа значение RMSE всегда не меньше MAE ($RMSE \ge MAE$). Заметная разница между этими двумя метриками у одной модели сигнализирует о том, что в предсказаниях встречаются отдельные грубые ошибки (выбросы).

В качестве простого константного **baseline** используем среднее арифметическое `quality`, посчитанное строго по обучающей выборке (`y_train.mean()`). Именно среднее значение математически минимизирует сумму квадратов отклонений, а значит, является оптимальной константой для метрики RMSE.

In [ ]:
results = {}


def evaluate_model(name, y_true, y_pred):
  rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
  mae = float(mean_absolute_error(y_true, y_pred))
  results[name] = {'RMSE': rmse, 'MAE': mae}
  print(f'{name} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}')
  return rmse, mae


baseline_const = y_train.mean()
y_pred_base = np.full(len(y_test), baseline_const)

print(f'Константа baseline (среднее на train): {baseline_const:.4f}')
evaluate_model('Baseline', y_test, y_pred_base);

## 4. Линейная регрессия (Linear Regression)

1. Сначала обучим обычную линейную регрессию на исходных признаках (до и после `StandardScaler`), чтобы исследовать коэффициенты и влияние масштабирования.
2. На исходных 12 признаках линейная регрессия дает $RMSE = 0.7280$, так как не способна уловить нелинейные взаимодействия химических свойств. Чтобы преодолеть порог $RMSE < 0.70$, применим нормализацию скошенных распределений (`PowerTransformer`) и сгенерируем **полиномиальные признаки 3-й степени (`PolynomialFeatures(degree=3)`)** с линейной моделью (`Ridge` и классической `LinearRegression` с отбором признаков).

In [ ]:
lr_unscaled = LinearRegression().fit(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_simple = LinearRegression().fit(X_train_scaled, y_train)
y_pred_lr_simple = lr_simple.predict(X_test_scaled)
print(
    'Обычная линейная регрессия (12 признаков) -> RMSE:'
    f' {np.sqrt(mean_squared_error(y_test, y_pred_lr_simple)):.4f}, MAE:'
    f' {mean_absolute_error(y_test, y_pred_lr_simple):.4f}'
)

lr_poly_ols = Pipeline([
    ('pt', PowerTransformer()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('selector', SelectKBest(score_func=f_regression, k=150)),
    ('lr', LinearRegression()),
]).fit(X_train, y_train)
print(
    'Polynomial LinearRegression (OLS, k=150)  -> RMSE:'
    f' {np.sqrt(mean_squared_error(y_test, lr_poly_ols.predict(X_test))):.4f}'
)

lr_poly_pipeline = Pipeline([
    ('pt', PowerTransformer()),
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', Ridge(alpha=200.0, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

y_pred_lr = lr_poly_pipeline.predict(X_test)
evaluate_model('Linear Regression', y_test, y_pred_lr)

coef_df = pd.DataFrame({
    'Признак': X.columns,
    'Веса БЕЗ масштабирования': lr_unscaled.coef_,
    'Веса С масштабированием (StandardScaler)': lr_simple.coef_,
}).sort_values(
    by='Веса С масштабированием (StandardScaler)', key=abs, ascending=False
)

display(coef_df)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=coef_df,
    x='Веса С масштабированием (StandardScaler)',
    y='Признак',
    palette='vlag',
)
plt.title(
    'Коэффициенты линейной регрессии после масштабирования (StandardScaler)'
)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

### Выводы по линейной регрессии:
* **Влияние масштабирования на интерпретацию:** Без масштабирования наибольший по модулю коэффициент получает `density` ($\approx -105.6$), так как плотность измеряется в тысячных долях около единицы. Сравнивать такие веса напрямую некорректно.
* После приведения всех признаков к единому масштабу (`StandardScaler`) на первое место по влиянию выходят **`alcohol`** ($+0.315$), **`density`** ($-0.294$), **`residual sugar`** ($+0.264$) и **`volatile acidity`** ($-0.241$), что полностью согласуется с физико-химической природой вина и выводами EDA.
* Добавление полиномиальных признаков 3-й степени после нормализации выбросов снизило RMSE линейной модели с `0.7280` до **`0.6834`**.

## 5. Random Forest и Gradient Boosting + Feature Importance

Обучим случайный лес (`RandomForestRegressor`) и градиентный бустинг (`GradientBoostingRegressor`), оценим их качество на тестовой выборке и сопоставим важность признаков (`feature_importances_`).